In [1]:
import numpy as np
import joblib
from sklearn.metrics import classification_report, roc_auc_score
from tensorflow import keras
import numpy as np

In [3]:
X_test = np.load('../data/X_test.npy', allow_pickle=True).astype('float32')
y_test = np.load('../data/y_test.npy', allow_pickle=True).astype('float32')

In [4]:
log_reg = joblib.load('../models/logistic_regression_baseline.pkl')
log_reg_test_proba = log_reg.predict_proba(X_test)[:,1]
log_reg_test_pred = (log_reg_test_proba > 0.5).astype(int)

In [6]:
print("=== Logistic Regression — TEST SET ===")
print(classification_report(y_test, log_reg_test_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, log_reg_test_proba):.4f}")

# Neural Network v2 @ 0.35 on test
model_final = keras.models.load_model('../models/neural_network.keras')
nn_test_proba = model_final.predict(X_test).ravel()
nn_test_pred = (nn_test_proba > 0.35).astype(int)

print("\n=== Neural Network v2 @ 0.35 — TEST SET ===")
print(classification_report(y_test, nn_test_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, nn_test_proba):.4f}")

=== Logistic Regression — TEST SET ===
              precision    recall  f1-score   support

         0.0       0.86      0.88      0.87       777
         1.0       0.64      0.59      0.61       280

    accuracy                           0.80      1057
   macro avg       0.75      0.74      0.74      1057
weighted avg       0.80      0.80      0.80      1057

ROC-AUC: 0.8552
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 

=== Neural Network v2 @ 0.35 — TEST SET ===
              precision    recall  f1-score   support

         0.0       0.89      0.81      0.85       777
         1.0       0.57      0.72      0.64       280

    accuracy                           0.78      1057
   macro avg       0.73      0.76      0.74      1057
weighted avg       0.81      0.78      0.79      1057

ROC-AUC: 0.8529


## Final Model Comparison (Test Set)

Both models were evaluated once on the held-out test set (1,057 samples, 
never used during training, validation, or threshold tuning) to produce 
an honest, final comparison.

### Results

| Metric              | Logistic Regression | Neural Network v2 @ 0.35 |
|----------------------|:--------------------:|:---------------------------:|
| Accuracy             | **0.80**             | 0.78                        |
| Churn Precision      | **0.64**             | 0.57                        |
| Churn Recall         | 0.59                 | **0.72**                    |
| Churn F1             | 0.61                 | **0.64**                    |
| ROC-AUC              | 0.8552               | 0.8529                      |

### Interpretation

On the test set, both models achieve nearly identical ROC-AUC (0.8552 vs. 
0.8529) — consistent with the validation-set finding from Phase 7. This 
confirms the earlier conclusion: the two models have essentially the same 
raw discriminative power, suggesting the relationships in this dataset 
(tenure, contract type, internet service, payment method) are largely 
linear, and the neural network's added complexity did not unlock 
significant additional non-linear signal.

Where the models diverge meaningfully is in the recall/precision trade-off 
produced by the neural network's tuned decision threshold (0.35, chosen in 
Phase 6). On unseen test data, the neural network catches **72% of actual 
churners**, compared to Logistic Regression's 59% — a substantial, and now 
confirmed, improvement in the metric most aligned with the business 
priority established in Phase 1. This comes at an accepted cost: precision 
drops from 0.64 to 0.57, meaning a higher proportion of flagged customers 
will not actually churn.

### Conclusion

Both models have comparable underlying discriminative ability. The neural 
network's real advantage comes not from its architecture alone, but from 
threshold tuning enabled by inspecting its calibrated probability outputs — 
a lever equally available to Logistic Regression, but only exploited here 
for the neural network. Given the project's stated priority of minimizing 
missed churners over minimizing false alarms, **the Neural Network v2 @ 
0.35 threshold is selected as the final model** for deployment.